# iGEM Teams Topic Hierarchy (Low → Mid → High)

Builds a three-level hierarchy for the existing **iGEM teams** topic model,
mirroring the approach used for papers.

- **mid** — target number of groups (scaled proportionally from papers; default 27)
- **high** — best k in a configurable range, selected by silhouette score

Inputs:
- `assets/topic_models/teams_topic_model`
- `assets/topic_models/teams_doc_topics.txt`
- `assets/topic_models/teams_topic_names.txt`
- `assets/embeddings/teams_corpus.txt`
- `assets/igem.txt`

Outputs:
- `assets/reports/teams_topic_hierarchy_map.tsv` — document-level mapping (UT, low, mid, high)
- `assets/reports/teams_topic_name_hierarchy.tsv` — topic names mapped to hierarchy (global_name, low, mid, high)
- `assets/reports/teams_topic_hierarchy_summary.tsv` — mid/high group summary stats

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from bertopic import BERTopic

from hierarchy_utils import (
    build_hierarchy_maps,
    select_best_high_k,
    build_topic_hierarchy_df,
    build_doc_map,
    build_name_map,
    build_summary,
)

SEED = 42
np.random.seed(SEED)

ROOT_DIR = Path.cwd().parent
ASSETS_DIR = ROOT_DIR / "assets"
MODELS_DIR = ASSETS_DIR / "topic_models"
EMBEDDINGS_DIR = ASSETS_DIR / "embeddings"
REPORTS_DIR = ASSETS_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "teams_topic_model"
DOC_TOPICS_PATH = MODELS_DIR / "teams_doc_topics.txt"
TOPIC_NAMES_PATH = MODELS_DIR / "teams_topic_names.txt"
TEAMS_CORPUS_PATH = EMBEDDINGS_DIR / "teams_corpus.txt"
IGEM_PATH = ASSETS_DIR / "igem.txt"

OUT_DOC_MAP = REPORTS_DIR / "teams_topic_hierarchy_map.tsv"
OUT_NAME_MAP = REPORTS_DIR / "teams_topic_name_hierarchy.tsv"
OUT_SUMMARY = REPORTS_DIR / "teams_topic_hierarchy_summary.tsv"

HIGH_K_MIN = 4
HIGH_K_MAX = 11
MID_K_TARGET = 27

In [8]:
# Load model and input datasets
print("Loading model and datasets …")
teams_topic_model = BERTopic.load(str(MODEL_PATH))

teams_doc_topics = pd.read_csv(DOC_TOPICS_PATH, sep="\t")
teams_topic_names = pd.read_csv(TOPIC_NAMES_PATH, sep="\t")
teams_corpus = pd.read_csv(TEAMS_CORPUS_PATH, sep="\t")
igem = pd.read_csv(IGEM_PATH, sep="\t")

# Normalize key column names
teams_doc_topics = teams_doc_topics.rename(columns={"topic": "low"})
teams_topic_names = teams_topic_names.rename(columns={"topic": "low"})

assert {"UT", "low"}.issubset(teams_doc_topics.columns)
assert {"low", "global_name"}.issubset(teams_topic_names.columns)
assert {"UT", "text"}.issubset(teams_corpus.columns)
assert {"UT", "Year_y"}.issubset(igem.columns)

teams_doc_topics["low"] = teams_doc_topics["low"].astype(int)
teams_topic_names["low"] = teams_topic_names["low"].astype(int)
igem["Year_y"] = pd.to_numeric(igem["Year_y"], errors="coerce")

print(f"  teams_doc_topics : {len(teams_doc_topics):,} rows")
print(f"  teams_topic_names: {len(teams_topic_names):,} rows")
print(f"  teams_corpus     : {len(teams_corpus):,} rows")
print(f"  igem             : {len(igem):,} rows")
print(f"  Outlier docs (low = -1): {(teams_doc_topics['low'] == -1).sum():,}")

Loading model and datasets …
  teams_doc_topics : 4,548 rows
  teams_topic_names: 161 rows
  teams_corpus     : 4,548 rows
  igem             : 4,548 rows
  Outlier docs (low = -1): 0


In [9]:
# Build hierarchy and select levels
print("Building hierarchy …")
corpus_texts = teams_corpus["text"].astype(str).tolist()

hier_df, maps_by_k, low_topics = build_hierarchy_maps(
    teams_topic_model, corpus_texts,
    max_k=max(HIGH_K_MAX, MID_K_TARGET),
)

# Ensure MID_K_TARGET exists in the map; fall back to nearest available
if MID_K_TARGET not in maps_by_k:
    MID_K_TARGET = max(k for k in maps_by_k if k >= HIGH_K_MAX)
    print(f"  Adjusted MID_K_TARGET → {MID_K_TARGET}")

selected_high_k, selected_high_score, candidate_scores = select_best_high_k(
    teams_topic_model, low_topics, maps_by_k,
    k_min=HIGH_K_MIN, k_max=HIGH_K_MAX,
)

low_to_mid = maps_by_k[MID_K_TARGET]
low_to_high = maps_by_k[selected_high_k]

topic_hierarchy_map = build_topic_hierarchy_df(low_topics, low_to_mid, low_to_high)

pd.DataFrame(candidate_scores, columns=["high_k", "silhouette"]).sort_values("high_k")

Building hierarchy …
  Building BERTopic hierarchical merge tree …


100%|██████████| 160/160 [00:01<00:00, 94.12it/s] 

  Non-outlier low topics: 161
  Tracking cluster maps for k = 1 … 27
  Captured 27 distinct k-level snapshots
  Scoring high-level k candidates in [4, 11] …
    k=  4  silhouette=0.0232
    k=  5  silhouette=0.0301
    k=  6  silhouette=0.0381
    k=  7  silhouette=0.0620
    k=  8  silhouette=0.0625
    k=  9  silhouette=0.0540
    k= 10  silhouette=0.0665
    k= 11  silhouette=0.0422
  ✓ Selected high-level k = 10 (silhouette = 0.0665)
  Hierarchy table: 161 low → 27 mid → 10 high


,high_k,silhouette
0,4,0.023189
1,5,0.030103
2,6,0.038137
3,7,0.062048
4,8,0.062452
5,9,0.053974
6,10,0.066455
7,11,0.042242


In [10]:
# Report 1: team-level mapping (UT, low, mid, high)
print("Building document map …")
report1 = build_doc_map(teams_doc_topics, topic_hierarchy_map, id_col="UT")
report1.to_csv(OUT_DOC_MAP, sep="\t", index=False)

print(f"  Saved → {OUT_DOC_MAP.name}")
report1.head()

Building document map …
  Document map: 4,548 rows (0 outliers)
  Saved → teams_topic_hierarchy_map.tsv


,UT,low,mid,high
0,5260,24,10,7
1,5794,114,20,5
2,4831,106,18,2
3,4068,17,2,2
4,2815,117,12,2


In [11]:
# Report 2: low-level global names mapped to low/mid/high
print("Building name map …")
report2 = build_name_map(teams_topic_names, topic_hierarchy_map)
report2.to_csv(OUT_NAME_MAP, sep="\t", index=False)

print(f"  Saved → {OUT_NAME_MAP.name}")
report2.head()

Building name map …
  Name map: 161 topics
  Saved → teams_topic_name_hierarchy.tsv


,global_name,low,mid,high
0,Synthetic Diagnostics & Biosensing Technologies,0,0,0
1,Light-Controlled Biological Systems,1,1,1
2,Synthetic Microbial Detection & Biofilm Manage...,2,2,2
3,Synthetic Plastic Degradation & Recycling,3,3,3
4,Synthetic Therapeutics for Cancer & Diseases,4,4,4


In [12]:
# Report 3: mid/high summaries with count, average year, median year
print("Building summary …")
report3 = build_summary(
    report1, igem,
    id_col="UT", year_col="Year_y",
)
report3.to_csv(OUT_SUMMARY, sep="\t", index=False)

print(f"  Saved → {OUT_SUMMARY.name}")
report3.head(10)

Building summary …
  Summary: 37 rows (mid + high)
  Saved → teams_topic_hierarchy_summary.tsv


,level,group_id,total_count,avg_publication_year,median_publication_year
0,high,0,302,2020.112583,2020.5
1,high,1,926,2016.193305,2016.0
2,high,2,575,2018.387826,2019.0
3,high,3,180,2020.861111,2022.0
4,high,4,254,2019.771654,2020.0
5,high,5,660,2019.843939,2021.0
6,high,6,521,2018.748560,2019.0
7,high,7,772,2018.603627,2019.0
8,high,8,138,2018.746377,2019.0
9,high,9,220,2019.727273,2020.0


In [13]:
# Validation
assert list(report1.columns) == ["UT", "low", "mid", "high"]
assert list(report2.columns) == ["global_name", "low", "mid", "high"]
assert {"level", "group_id", "total_count", "avg_publication_year", "median_publication_year"}.issubset(report3.columns)
assert len(report1) == len(teams_doc_topics)
assert (report1.loc[report1["low"] == -1, ["mid", "high"]] == -1).all().all()
assert HIGH_K_MIN <= selected_high_k <= HIGH_K_MAX

print("All validation checks passed ✓")
print(f"  report1: {len(report1):,} rows | report2: {len(report2):,} rows | report3: {len(report3):,} rows")

All validation checks passed ✓
  report1: 4,548 rows | report2: 161 rows | report3: 37 rows


In [14]:
# Hierarchy quality summary
quality = pd.DataFrame(candidate_scores, columns=["high_k", "silhouette"]).sort_values("high_k")
outlier_pct = (teams_doc_topics["low"] == -1).mean() * 100

print(f"Selected high-level K : {selected_high_k}")
print(f"Selected silhouette   : {selected_high_score:.4f}")
print(f"Outlier documents     : {outlier_pct:.2f}%")

quality

Selected high-level K : 10
Selected silhouette   : 0.0665
Outlier documents     : 0.00%


,high_k,silhouette
0,4,0.023189
1,5,0.030103
2,6,0.038137
3,7,0.062048
4,8,0.062452
5,9,0.053974
6,10,0.066455
7,11,0.042242
